# Notebook 09 — Backtesting Engine

**Phase 4 · Evaluation & Deployment (1 / 2)**

---

## 🎯 Learning Objectives

| # | Goal |
|---|------|
| 1 | Build a simple **walk-forward backtester** from scratch |
| 2 | Implement key performance metrics: **Sharpe, Sortino, Calmar, Profit Factor, Win Rate** |
| 3 | Understand **train / validation split** for strategy evaluation |
| 4 | Visualize equity curves, drawdowns, and rolling Sharpe |
| 5 | Compare hand-built metrics with the production `core_module_backtester.py` |

### Prerequisites
- NB03 (Risk Metrics), NB04–NB08 (strategies and risk pipeline)

In [ ]:
# ── Setup ──────────────────────────────────────────────────
import sys, pathlib, warnings, math
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
DAYS_PER_YEAR = 365
print("✅ Imports OK  |  Project root:", ROOT)

---
## 1 · Backtesting Fundamentals

A backtest answers: **"If I had run this strategy in the past, what would have happened?"**

### Walk-Forward Protocol

```
          ┌── Train (90d) ──┐┌── Validation (90d) ──┐
timeline: ═══════════════════════════════════════════▶
          ^                  ^                       ^
       start           split point                 end
```

- **Train period:** Optimize parameters, learn patterns
- **Validation period:** Evaluate on unseen data (no peeking!)
- The production backtester uses `history_days=180`, `train_days=90`, `validation_days=90`

### Key Metrics

| Metric | Formula | What it Tells You |
|--------|---------|-------------------|
| **Sharpe** | $\frac{\bar{r}}{\sigma_r} \sqrt{N}$ | Risk-adjusted return (penalizes all volatility) |
| **Sortino** | $\frac{\bar{r}}{\sigma_{\text{down}}} \sqrt{N}$ | Like Sharpe but only penalizes downside |
| **Calmar** | $\frac{\text{CAGR}}{|\text{MaxDD}|}$ | Return per unit of worst-case drawdown |
| **Profit Factor** | $\frac{\sum \text{wins}}{|\sum \text{losses}|}$ | Gross profits / gross losses |
| **Win Rate** | $\frac{\text{winning days}}{\text{total days}}$ | % of days with positive returns |

---
## 2 · Synthetic Multi-Asset Backtest Data

In [ ]:
np.random.seed(42)
N_DAYS = 180
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=N_DAYS, freq="D")

# Generate correlated multi-asset returns
assets = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]
n_assets = len(assets)

# Market factor + idiosyncratic
market_factor = np.random.normal(0.001, 0.015, N_DAYS)
returns_dict = {}
for i, asset in enumerate(assets):
    beta = 0.5 + 0.5 * i / n_assets   # BTC has lowest beta, XRP highest
    alpha = 0.0005 * (n_assets - i) / n_assets  # BTC has highest alpha
    idio = np.random.normal(0, 0.01, N_DAYS)
    returns_dict[asset] = alpha + beta * market_factor + idio

returns_df = pd.DataFrame(returns_dict, index=dates)

# Build price panels
prices = pd.DataFrame(index=dates)
base_prices = {"BTCUSDT": 60000, "ETHUSDT": 3500, "SOLUSDT": 150, "BNBUSDT": 600, "XRPUSDT": 0.60}
for asset in assets:
    prices[asset] = base_prices[asset] * np.exp(np.cumsum(returns_df[asset]))

print(f"Price panel: {prices.shape[0]} days × {prices.shape[1]} assets")
print(f"Date range: {dates[0].date()} to {dates[-1].date()}")
prices.head(3)

---
## 3 · Performance Metrics from Scratch

In [ ]:
def annualized_sharpe(returns: pd.Series, periods_per_year: int = 365) -> float | None:
    """Annualized Sharpe ratio (assumes zero risk-free rate)."""
    r = returns.dropna().astype(float)
    if len(r) < 2:
        return None
    vol = float(r.std(ddof=0))
    if vol == 0:
        return None
    return float((r.mean() / vol) * math.sqrt(periods_per_year))

def annualized_sortino(returns: pd.Series, periods_per_year: int = 365) -> float | None:
    """Sortino ratio — only penalizes downside volatility."""
    r = returns.dropna().astype(float)
    if r.empty:
        return None
    downside = r[r < 0]
    dd = float(downside.std(ddof=0)) if not downside.empty else 0.0
    if dd == 0:
        return None
    return float((r.mean() / dd) * math.sqrt(periods_per_year))

def max_drawdown(returns: pd.Series) -> float | None:
    """Maximum drawdown from equity curve."""
    r = returns.dropna().astype(float)
    if r.empty:
        return None
    equity = (1 + r).cumprod()
    peak = equity.cummax()
    dd = (equity / peak) - 1
    return float(dd.min())

def calmar_ratio(returns: pd.Series, periods_per_year: int = 365) -> float | None:
    """Calmar = annualized return / |max drawdown|."""
    r = returns.dropna().astype(float)
    mdd = max_drawdown(r)
    if mdd is None or mdd == 0:
        return None
    ann_return = float(r.mean() * periods_per_year)
    return ann_return / abs(mdd)

def profit_factor(returns: pd.Series) -> float | None:
    """Gross profits / gross losses."""
    r = returns.dropna().astype(float)
    gross_profit = float(r[r > 0].sum())
    gross_loss = abs(float(r[r < 0].sum()))
    if gross_loss == 0:
        return None
    return gross_profit / gross_loss

def win_rate(returns: pd.Series) -> float | None:
    """Fraction of positive-return periods."""
    r = returns.dropna().astype(float)
    if r.empty:
        return None
    return float((r > 0).mean())

print("Metric functions defined ✅")

---
## 4 · A Simple Walk-Forward Backtester

In [ ]:
def simple_momentum_backtest(
    prices: pd.DataFrame,
    lookback: int = 5,
    top_n: int = 3,
) -> pd.Series:
    """
    Walk-forward backtest of a simple momentum strategy:
    - Each day, rank assets by lookback-day return
    - Go equal-weight long the top_n
    - Hold for 1 day, then rebalance
    """
    daily_returns = prices.pct_change()
    lookback_returns = prices.pct_change(lookback)
    
    strategy_returns = []
    strategy_dates = []
    
    for i in range(lookback, len(prices) - 1):
        # Rank by lookback return
        scores = lookback_returns.iloc[i].dropna()
        if len(scores) < top_n:
            continue
        top_assets = scores.nlargest(top_n).index.tolist()
        
        # Equal-weight next-day return
        next_day_ret = daily_returns.iloc[i + 1][top_assets].mean()
        strategy_returns.append(next_day_ret)
        strategy_dates.append(prices.index[i])
    
    return pd.Series(strategy_returns, index=strategy_dates, name="momentum")

# Run backtest
strat_returns = simple_momentum_backtest(prices, lookback=5, top_n=3)

# Benchmark: equal-weight buy-and-hold
benchmark_returns = prices.pct_change().mean(axis=1).iloc[5:]
benchmark_returns = benchmark_returns.reindex(strat_returns.index)

print(f"Strategy has {len(strat_returns)} daily returns")
print(f"First: {strat_returns.index[0].date()}  Last: {strat_returns.index[-1].date()}")

---
## 5 · Train / Validation Split

In [ ]:
split_point = len(strat_returns) // 2
train_returns = strat_returns.iloc[:split_point]
val_returns = strat_returns.iloc[split_point:]

train_bench = benchmark_returns.iloc[:split_point]
val_bench = benchmark_returns.iloc[split_point:]

def report_metrics(returns: pd.Series, label: str) -> dict:
    """Compute and print all metrics."""
    metrics = {
        "Sharpe": annualized_sharpe(returns),
        "Sortino": annualized_sortino(returns),
        "Calmar": calmar_ratio(returns),
        "Max DD": max_drawdown(returns),
        "Profit Factor": profit_factor(returns),
        "Win Rate": win_rate(returns),
        "Avg Daily Return": float(returns.mean()) if not returns.empty else 0,
    }
    print(f"\n{'='*40}")
    print(f" {label} ({len(returns)} days)")
    print(f"{'='*40}")
    for name, val in metrics.items():
        if val is None:
            print(f"  {name:18s}: N/A")
        elif "Rate" in name or "DD" in name:
            print(f"  {name:18s}: {val:.2%}")
        else:
            print(f"  {name:18s}: {val:.4f}")
    return metrics

train_metrics = report_metrics(train_returns, "TRAIN — Momentum")
val_metrics = report_metrics(val_returns, "VALIDATION — Momentum")
bench_metrics = report_metrics(benchmark_returns, "BENCHMARK — Equal Weight B&H")

---
## 6 · Equity Curves & Drawdown Visualization

In [ ]:
strat_equity = (1 + strat_returns).cumprod()
bench_equity = (1 + benchmark_returns.fillna(0)).cumprod()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})

# Equity curves
ax1.plot(strat_equity.index, strat_equity, label="Momentum Strategy", lw=1.5, color="#3498db")
ax1.plot(bench_equity.index, bench_equity, label="Equal-Weight B&H", lw=1.5, color="#95a5a6")

# Mark train/validation split
split_date = strat_returns.index[split_point]
ax1.axvline(split_date, ls="--", color="black", lw=1, label="Train/Val Split")
ax1.fill_between(strat_equity.index[:split_point+1], 0, strat_equity.max() * 1.1,
                 alpha=0.03, color="blue")
ax1.fill_between(strat_equity.index[split_point:], 0, strat_equity.max() * 1.1,
                 alpha=0.03, color="green")

ax1.set_ylabel("Equity")
ax1.set_title("Walk-Forward Backtest: Momentum Strategy")
ax1.legend(loc="upper left")

# Drawdown
strat_peak = strat_equity.cummax()
strat_dd = (strat_equity / strat_peak) - 1
ax2.fill_between(strat_dd.index, strat_dd, alpha=0.4, color="#e74c3c")
ax2.axvline(split_date, ls="--", color="black", lw=1)
ax2.set_ylabel("Drawdown")
ax2.set_xlabel("Date")
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

---
## 7 · Rolling Sharpe Ratio

In [ ]:
def rolling_sharpe(returns: pd.Series, window: int = 30) -> pd.Series:
    """Rolling annualized Sharpe using a fixed window."""
    rolling_mean = returns.rolling(window).mean()
    rolling_std = returns.rolling(window).std(ddof=0)
    return (rolling_mean / rolling_std) * math.sqrt(DAYS_PER_YEAR)

rs_strat = rolling_sharpe(strat_returns, window=30)
rs_bench = rolling_sharpe(benchmark_returns, window=30)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rs_strat.index, rs_strat, label="Momentum (30d rolling)", lw=1.5, color="#3498db")
ax.plot(rs_bench.index, rs_bench, label="Benchmark (30d rolling)", lw=1.5, color="#95a5a6")
ax.axhline(0, ls="-", color="black", lw=0.5)
ax.axhline(1.0, ls="--", color="green", lw=0.8, label="Sharpe = 1.0")
ax.axhline(-1.0, ls="--", color="red", lw=0.8, label="Sharpe = -1.0")
ax.axvline(split_date, ls="--", color="black", lw=1)
ax.set_ylabel("Rolling Sharpe")
ax.set_title("30-Day Rolling Sharpe Ratio")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 8 · Composite Score (Production Scoring)

The bot ranks strategy performance using a **composite score**:

$$
\text{Score} = 0.40 \times \text{Sortino} + 0.30 \times \text{Sharpe} + 0.30 \times \text{Calmar}
$$

In [ ]:
def composite_score(returns: pd.Series) -> float:
    """Weighted composite of Sortino (40%), Sharpe (30%), Calmar (30%)."""
    sortino = annualized_sortino(returns) or 0.0
    sharpe = annualized_sharpe(returns) or 0.0
    calmar = calmar_ratio(returns) or 0.0
    return 0.40 * sortino + 0.30 * sharpe + 0.30 * calmar

print(f"Composite Score (Train):      {composite_score(train_returns):.4f}")
print(f"Composite Score (Validation): {composite_score(val_returns):.4f}")
print(f"Composite Score (Benchmark):  {composite_score(benchmark_returns):.4f}")

---
## 9 · Parameter Sensitivity Sweep

In [ ]:
# Sweep lookback and top_n on TRAIN period only
results = []
for lookback in [3, 5, 7, 10, 14]:
    for top_n in [1, 2, 3, 4]:
        r = simple_momentum_backtest(prices, lookback=lookback, top_n=top_n)
        train_r = r.iloc[:len(r)//2]
        val_r = r.iloc[len(r)//2:]
        results.append({
            "lookback": lookback,
            "top_n": top_n,
            "train_sharpe": annualized_sharpe(train_r) or 0.0,
            "val_sharpe": annualized_sharpe(val_r) or 0.0,
            "train_score": composite_score(train_r),
            "val_score": composite_score(val_r),
        })

sweep_df = pd.DataFrame(results)

# Heatmap: train Sharpe
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (ax1, "train_sharpe", "Train Sharpe"),
    (ax2, "val_sharpe", "Validation Sharpe"),
]:
    pivot = sweep_df.pivot(index="lookback", columns="top_n", values=col)
    im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto",
                   vmin=-2, vmax=4)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("top_n")
    ax.set_ylabel("lookback")
    ax.set_title(title)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.values[i,j]:.1f}", ha="center", va="center", fontsize=9)

fig.suptitle("Parameter Sensitivity: Momentum Backtest", fontsize=13)
plt.tight_layout()
plt.show()

# Best parameters (by train composite score)
best = sweep_df.loc[sweep_df["train_score"].idxmax()]
print(f"\nBest params (by train score): lookback={int(best['lookback'])}, top_n={int(best['top_n'])}")
print(f"  Train score: {best['train_score']:.4f}  |  Val score: {best['val_score']:.4f}")

---
## 10 · Production Metrics Comparison

In [ ]:
# Import production metric functions
from bot.backtest.core_module_backtester import (
    _annualized_sharpe as prod_sharpe,
    _annualized_sortino as prod_sortino,
    _max_drawdown as prod_max_dd,
    _profit_factor as prod_pf,
)

test_returns = strat_returns

comparisons = [
    ("Sharpe", annualized_sharpe(test_returns), prod_sharpe(test_returns, periods_per_year=365)),
    ("Sortino", annualized_sortino(test_returns), prod_sortino(test_returns, periods_per_year=365)),
    ("Max DD", max_drawdown(test_returns), prod_max_dd(test_returns)),
    ("Profit Factor", profit_factor(test_returns), prod_pf(test_returns)),
]

print(f"{'Metric':18s} {'Ours':>10s} {'Production':>12s} {'Match':>6s}")
print("-" * 50)
for name, ours, prod in comparisons:
    if ours is None or prod is None:
        match = ours is None and prod is None
        print(f"{name:18s} {'N/A':>10s} {'N/A':>12s} {'✅' if match else '❌':>6s}")
    else:
        match = abs(ours - prod) < 1e-6
        print(f"{name:18s} {ours:10.6f} {prod:12.6f} {'✅' if match else '❌':>6s}")

---
## 11 · Return Distribution Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(strat_returns.dropna(), bins=40, alpha=0.7, color="#3498db", label="Strategy", density=True)
ax1.hist(benchmark_returns.dropna(), bins=40, alpha=0.5, color="#95a5a6", label="Benchmark", density=True)
ax1.axvline(0, ls="-", color="black", lw=0.5)
ax1.axvline(strat_returns.mean(), ls="--", color="#3498db", lw=1.5)
ax1.set_xlabel("Daily Return")
ax1.set_ylabel("Density")
ax1.set_title("Return Distribution")
ax1.legend()

# QQ-style: cumulative returns
sorted_strat = np.sort(strat_returns.dropna())
sorted_bench = np.sort(benchmark_returns.dropna())
min_len = min(len(sorted_strat), len(sorted_bench))
ax2.scatter(sorted_bench[:min_len], sorted_strat[:min_len], s=10, alpha=0.5, color="#3498db")
ax2.plot([-0.05, 0.05], [-0.05, 0.05], ls="--", color="red", lw=1)
ax2.set_xlabel("Benchmark Quantiles")
ax2.set_ylabel("Strategy Quantiles")
ax2.set_title("QQ Plot: Strategy vs Benchmark")

plt.tight_layout()
plt.show()

---
## 12 · Key Takeaways

| Concept | Detail |
|---------|--------|
| **Walk-Forward** | Train on first half, validate on second — no look-ahead bias |
| **Sharpe** | Risk-adjusted return; > 1.0 is good, > 2.0 is excellent |
| **Sortino** | Like Sharpe but only penalizes downside (more relevant for trading) |
| **Calmar** | CAGR ÷ MaxDD — measures return per unit of worst pain |
| **Profit Factor** | > 1.0 means gross wins > gross losses |
| **Composite Score** | 40% Sortino + 30% Sharpe + 30% Calmar |
| **Parameter Sweep** | Always check if train performance carries to validation |

### Backtesting Pitfalls to Avoid

| Pitfall | Mitigation |
|---------|------------|
| Look-ahead bias | Only use data available at decision time |
| Survivorship bias | Include delisted assets |
| Overfitting | Keep parameter space small; validate out-of-sample |
| Transaction costs | Include slippage and fees in returns |
| Data snooping | Pre-register your hypothesis before running the backtest |

---
## 🔬 Exercises

1. **Mean-Reversion Backtest:** Replicate Section 4 but buy the *worst* N performers each day. Compare Sharpe with momentum.

2. **Walk-Forward Optimization:** Implement a rolling 60-day train window that re-optimizes `lookback` and `top_n` every 30 days. Does this improve validation Sharpe?

3. **Transaction Cost Model:** Add a 10bp (0.1%) round-trip cost per trade. How does it affect profit factor?

4. **Regime-Conditional Backtest:** Split the equity curve by regime (from NB06) and compute Sharpe separately for bull, ranging, and bear periods.

---
## ✅ Knowledge Check

1. Why is the Sortino ratio more appropriate than Sharpe for trading strategies?
2. What does a profit factor of exactly 1.0 mean?
3. Why might train Sharpe = 3.0 but validation Sharpe = 0.5? What does this indicate?
4. How does the composite score weight each metric, and why?
5. Name three sources of look-ahead bias in backtesting.

---
## 🔗 Next

**[NB10 — Full Pipeline Integration →](10_Full_Pipeline_Integration.ipynb)**

We'll wire everything together: data → signals → regime → ensemble → portfolio → risk → execution, replicating the production `main.py` trading loop.